# Agentic RAG Usando CrewAI & LangChain

Veremos como os agentes podem se envolver no sistema RAG para obter as informações mais relevantes.

In [ ]:
!pip install crewai crewai_tools langchain_openai langchain_community --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.9/211.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 514.9/514.9 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.8/131.8 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.4/211.4 kB 13.9 M

In [ ]:
from langchain_openai import ChatOpenAI
import os
from crewai_tools import PDFSearchTool
from langchain_community.tools.tavily_search import TavilySearchResults
from crewai import Crew
from crewai import Task
from crewai import Agent
from crewai.tools import tool

## Armazenar na variável de ambiente a OPEN API Key

In [ ]:
import os

# Set the API key
os.environ['OPENAI_API_KEY'] = ''


## Carregar o LLM que será usado

In [ ]:
llm = ChatOpenAI(
    openai_api_base="https://api.openai.com/v1/chat/completions",
    openai_api_key=os.environ['OPENAI_API_KEY'],
    model_name="gpt-4o-mini",
    temperature=0.1,
    max_tokens=1000,
)

## Criar uma ferramenta RAG para buscar conteúdo no PDF

## Armazenar na variável de ambiente Tavily API Key

In [ ]:
import os

# Set the Tavily API key
os.environ['TAVILY_API_KEY'] = 'sua chave'

In [ ]:
web_search_tool = TavilySearchResults(k=3)

In [ ]:
web_search_tool.run("Taxação do PIX")

[{'url': 'https://valorinveste.globo.com/produtos/servicos-financeiros/noticia/2023/06/20/pix-vai-ser-taxado-qual-o-valor-entenda-a-decisao-da-caixa-sobre-a-tarifa.ghtml',
  'content': 'A cobrança da tarifa do Pix para clientes pessoa jurídica é permitida pelo Banco Central e adotada por vários bancos A cobrança de tarifa do Pix, anunciada ontem (19) pela Caixa Econômica Federal, pegou muita gente de surpresa e reacendeu o debate político nas redes sociais dizendo que o presidente Lula vai taxar o Pix. A tarifa do Pix para pessoas jurídicas, no entanto, não é novidade. Se o recebedor for pessoa jurídica e usar Pix informando os dados da conta ou chave. Quanto ao valor da tarifa que é cobrado pela transação financeira, o Banco Central não define uma tarifa fixa ou um limite. A taxa para o pix compra será de 0,89% do valor da operação, com mínimo de R$ 1,00 e máximo de R$ 130,00.'},
 {'url': 'https://extra.globo.com/economia/noticia/2025/01/receita-federal-nega-taxacao-do-pix-e-esclarece

## Define uma Ferramenta(Tool)

In [ ]:
from crewai.tools import tool
@tool("BuscadorWeb")
def my_tool(tema: str) -> str:
    """Realiza busca na web de reportagens que abordam o tema da notícia."""
    # Function logic here
    return web_search_tool.run(tema)



## Cria Agentes para trabalhar

In [ ]:
Retriever_Agent = Agent(
role="Recuperador",
goal="Responsável por recuperar ou coletar informações de fontes de notícias e blogs",
backstory=(
    "Você é um agente responsável por coletar informações atuais de fontes de notícias e blogs."
    "Use as informações presente no contexto recuperado para fins de classifição do tom das reportagens (positiva, negativa, neutro)."
    "Para isso execute uma pesquisa na web usando uma ferramenta de pesquisa em páginas web."
),
verbose=True,
allow_delegation=False,
llm=llm,
)

In [ ]:
Grader_Retriever_Agent =  Agent(
  role='Avaliador de Recuperação',
  goal='Filtrar recuperação errônea',
  backstory=(
    "Você é um avaliador da relevância do conteúdo recuperado para uma tema de notícia."
    "Se o conteúdo da reportagem conter palavras chaves relacionadas ao tema da notícia, avalie ela como relevante."
  ),
  verbose=True,
  allow_delegation=False,
  llm=llm,
)

In [ ]:
Classify_Agent = Agent(
    role="Responsável por classificar o tom das reportagens..",
    goal="Classificar o tom das reportagens (positiva, negativa, neutra).",
    backstory=(
        "Você é um classificador de noticias. Você deve classificar se o tom do conteúdo de cada reportagem é positiva, negativa, neutra."
        "Certifique-se de revisar meticulosamente cada reportagem e verificar se ela é do tipo positiva, negativa, neutra."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

In [ ]:
Calculator_Agent = Agent(
    role="Responsável por calcular a porcetagem de tom das reportagens.",
    goal="Calcular a porcentagem de tom das reportagens (positiva, negativa, neutra).",
    backstory=(
        "Você é um estatístico. Você deve calcular a porcentagem de prevalência de classes de tom das notícias."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

## Definir Tasks (Tarefas)

In [ ]:
retriever_task = Task(
    description=("Use uma ferramenta de pesquisa na web para recuperar as reportagens mais recentes de sites, páginas web que sejam relacionadas ao tema {tema} de notícia."
    ),
    expected_output=("Você deverá usar uma ferramenta de pesquisa na web para recuperar reportagens de sites, páginas web que sejam relacionadas ao tema {tema} de notícia."
    "Retorne um texto claro e consiso como resposta."),
    agent=Retriever_Agent,
    tools=[my_tool],
)

In [ ]:
grader_retriever_task = Task(
    description=("Baseado na resposta da tarefa do 'retriever task' para o tema {tema} da nóticia avalie se o conteúdo recuperado de cada reportagem tem relação com o tema da notícia."
    ),
    expected_output=("Realizar uma nova pesquisa na web por meio 'retriever task'  se a resposta do 'retriever_task' não possui relação com o tema, caso contrário retornar cada reportagem recuperada." ),
    agent=Grader_Retriever_Agent,
    context=[retriever_task],

)

In [ ]:
classify_task = Task(
    description=("Baseado na resposta do  'grader_retriever_task' classifique se o conteúdo de cada reportagem é positiva (Notícias que trazem boas notícias, esperança, celebrações ou conquistas), negativa (Notícias que comunicam tragédias, problemas ou situações preocupantes), neutra (Notícias que apresentam fatos sem emoção explícita ou tendem a ser objetivas)."
    ),
    expected_output=("Classificação do tom de cada reportagem e explicação do motivo da classificação."),
    context=[grader_retriever_task],
    agent=Classify_Agent,
)

In [ ]:
calculator_task = Task(
    description=("Baseado na resposta do 'classify_task' calcule a porcentagem de reportagens que foram classificadas como positiva, negativa, neutra."
    ),
    expected_output=("Com base na resposta do 'classify_task' calcule a porcentagem de reportagens que foram classificadas como positiva, negativa, neutra."),
    context=[classify_task],
    agent=Calculator_Agent,
)

## Define um fluxo para o caso de uso


In [ ]:
rag_crew = Crew(
    agents=[Retriever_Agent, Grader_Retriever_Agent, Classify_Agent, Calculator_Agent],
    tasks=[retriever_task, grader_retriever_task, classify_task, calculator_task],
    verbose=True,
)

## Qual é o tema da noíticia?


In [ ]:
inputs ={"tema":"Flamengo"}

## Kick off the agent pipeline

In [ ]:
result = rag_crew.kickoff(inputs=inputs)

# Agent: Recuperador
## Task: Use uma ferramenta de pesquisa na web para recuperar as reportagens mais recentes de sites, páginas web que sejam relacionadas ao tema Flamengo de notícia.


# Agent: Recuperador
## Using tool: BuscadorWeb
## Tool Input: 
"{\"tema\": \"Flamengo\"}"
## Tool Output: 
[{'url': 'https://www.sofascore.com/team/football/flamengo/5981', 'content': 'Flamengo scores, fixtures, standings and player stats\nFlamengo\nDetails\nMatches\nSquad\nAbout Flamengo\nFlamengo live scores, players, season schedule and today’s results are available on Sofascore.\n Thiago Maia\nMidfielder: Bruno Henrique, Everton Ribeiro, Igor Jesus Lima, Lorran\nDefender: Filipe Luís, Rodrigo Caio, Léo Pereira, Fabrício Bruno\nUnknown: Gabriel Barbosa, Giorgian De Arrascaeta, Pedro, Gerson, Everton, David Luiz, Erick Pulgar, Ayrton Lucas, Luiz Araújo, Matheus Cunha, Victor Hugo, Matheuzinho, Santos, Agustín Rossi, Allan, Guillermo Varela, Wesley França, Pablo, Gabriel Noga, Thiaguinho, Cleiton Sa

## Mostra a resposta final

In [ ]:
print(result)

Para calcular a porcentagem de tom das reportagens classificadas como positiva, negativa e neutra, vamos analisar as classificações fornecidas:

1. Sofascore: Negativa
2. Wikipedia: Neutra
3. FIFA: Positiva
4. ESPN: Negativa
5. Globo Esporte: Positiva

Total de reportagens: 5

Classificações:
- Positivas: 2 (FIFA e Globo Esporte)
- Negativas: 2 (Sofascore e ESPN)
- Neutras: 1 (Wikipedia)

Agora, vamos calcular as porcentagens:

- Porcentagem de reportagens positivas: (2/5) * 100 = 40%
- Porcentagem de reportagens negativas: (2/5) * 100 = 40%
- Porcentagem de reportagens neutras: (1/5) * 100 = 20%

Portanto, as porcentagens de tom das reportagens são:
- Positivas: 40%
- Negativas: 40%
- Neutras: 20%

Esses resultados refletem a prevalência das diferentes classes de tom nas reportagens analisadas.
